In [ ]:
from netgen.meshing import Mesh
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from netgen.occ import *
from ngsolve.webgui import Draw

import matplotlib.pylab as plt
import numpy as np

In [ ]:
def Capacitor3DGeometry(box_size, W, L, d, D, h_max):
    air_box = Box((-box_size, -box_size, -box_size), (box_size, box_size, box_size))
    air_box.faces.name = "Outer"  

    dielectric = Box((-W/2, -d/2, -L/2), (W/2, d/2, L/2))
    dielectric.faces.maxh = h_max/2
    dielectric.name = "dielectric"

    electrode_positive = Box((-W/2, d/2, -L/2), (W/2, (d+D)/2, L/2))
    electrode_positive.faces.maxh = h_max/4
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = Box((-W/2, -(d+D)/2, -L/2), (W/2, -d/2, L/2))
    electrode_negative.faces.maxh = h_max/4
    electrode_negative.faces.name = "electrode_negative"

    air = air_box - dielectric
    air.name = "air"

    shape = Glue([air, dielectric])
    shape = shape - electrode_positive - electrode_negative

    return shape


def Capacitor3DMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=3).GenerateMesh(maxh=h_max))

    return mesh


def Capacitor3DWeakForm(mesh, FE_order, epsr):

    fes_phi = H1(mesh, order=FE_order, dirichlet="el.*")
    fes_E = HCurl(mesh, order=FE_order-1)
    fes_D = HDiv(mesh, order=FE_order-1)
    fes_rho = L2(mesh, order=FE_order-2)

    u = fes_phi.TrialFunction()
    v = fes_phi.TestFunction()

    a = BilinearForm(epsr*grad(u)*grad(v)*dx)
    precond = preconditioners.Local(a)

    return a, precond, fes_phi, fes_E, fes_D, fes_rho


def Capacitor3DAssemble(a):
     
    pre = preconditioners.Local(a)
     
    with TaskManager():
        a.Assemble()

    return a
    

def Capacitor3DSolver(mesh, fes, a, precond):

    potential_gf = GridFunction(fes)
    potential_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))
        

    with TaskManager():
        inv = CGSolver(
            mat=a.mat,
            pre=precond,
            printrates='\r',
            maxiter=10000
        )
      
        potential_gf.vec.data -= inv*(a.mat * potential_gf.vec)

    return potential_gf

In [ ]:
clipping = {"function": True, "pnt": (0, 0, 0), "vec": (0, 0, -1)}

In [ ]:
box_size, W, L, d, D = 15, 5, 5, 1.5, 0.5
epsr_air, epsr_dielectric = 1.0, 4.0

FE_order = 2
h_max = 2

geo = Capacitor3DGeometry(box_size, W, L, d, D, h_max)

In [ ]:
mesh = Capacitor3DMesh(geo, h_max)

In [ ]:
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

In [ ]:
Draw(mesh, clipping=clipping, settings={"Objects": {"Surface": True}});

In [ ]:
a, precond, fes_phi, fes_E, fes_D, fes_rho = Capacitor3DWeakForm(mesh, FE_order, epsr)

In [ ]:
a = Capacitor3DAssemble(a)

In [ ]:
phi_gf = Capacitor3DSolver(mesh, fes_phi, a, precond)

In [ ]:
E_gf = GridFunction(fes_E)
D_gf = GridFunction(fes_D)
rho_gf = GridFunction(fes_rho)

E_gf.Set(-grad(phi_gf))
D_gf.Set(epsr*E_gf)
rho_gf.Set(div(D_gf))

n = specialcf.normal(mesh.dim)

In [ ]:
N = 20
margin = 10
y_start = d/2 - 1e-3
x_min, x_max = -W/2 - margin, W/2 + margin
z_min, z_max = -L/2 - margin, 0


p = [(
     x_min + (x_max - x_min)*i/N,
     y_start,
     z_min + (z_max - z_min)*j/N
    )
    for i in range(N)
    for j in range(N)
]

fieldlines = E_gf._BuildFieldLines(mesh, p, num_fieldlines=400, length=6)

Draw(E_gf, mesh, "Electric field E", 
     draw_vol=True, 
     draw_surf=True, 
     objects=[fieldlines],
     autoscale=True, 
     min = 0, 
     max = 1, 
     settings={"Objects": {"Surface": False}}
     );

In [ ]:
energy_E = 0.5 * Integrate(epsr*InnerProduct(E_gf, E_gf), mesh, definedon=mesh.Materials("dielectric"))
energy_all = 0.5 * Integrate(epsr*InnerProduct(E_gf, E_gf), mesh)
energy_gradphi = 0.5 * Integrate(epsr*InnerProduct(grad(phi_gf), grad(phi_gf)), mesh, definedon=mesh.Materials("dielectric"))
energy_formula = 0.5 * epsr_dielectric*W*L/d*4
print ("Energy from E field in the dielectric:", energy_E)
print ("Energy from grad(phi) in the dielectric:", energy_gradphi)
print ("Energy in the whole space:", energy_all)
print ("Energy by OE formula:", energy_formula)

In [ ]:
Q= Integrate(rho_gf, mesh)
print("Total charge in the environment =", Q)

In [ ]:
Draw(rho_gf, mesh, min=0, max=1, clipping=clipping);

In [ ]:
Q_outer = Integrate(D_gf*n, mesh, definedon=mesh.Boundaries("Outer"))
Q_pos = Integrate(D_gf*n, mesh, definedon=mesh.Boundaries("electrode_positive"))
Q_neg = Integrate(D_gf*n, mesh, definedon=mesh.Boundaries("electrode_negative"))

print("Q_outer =", Q_outer)
print("Q_pos   =", Q_pos)
print("Q_neg   =", Q_neg)
print("sum     =", Q_outer + Q_pos + Q_neg)
print("int div D =", Integrate(rho_gf, mesh))